# 06 — Audit Report Examples

**Rift: Graph ML for Fraud Detection, Replay, and Audit**

This notebook generates plain-English audit reports, demonstrates deterministic replay, and shows the full decision lifecycle from prediction through explainability to governance.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AngelP17/Rift/blob/main/notebooks/06_audit_examples.ipynb)


## Environment Setup


In [ ]:
# Clone repo and install dependencies (run once)
import subprocess, sys, os

REPO = "https://github.com/AngelP17/Rift.git"
if not os.path.exists("/content/Rift"):
    subprocess.run(["git", "clone", "--depth", "1", REPO, "/content/Rift"], check=True)

os.chdir("/content/Rift")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "polars", "numpy", "pandas", "scikit-learn", "xgboost", "duckdb",
    "shap", "structlog", "python-dotenv", "rich", "jinja2", "pyarrow"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "torch", "--index-url", "https://download.pytorch.org/whl/cpu"], check=False)
sys.path.insert(0, "src")

try:
    import torch
    device = "cuda" if torch.cuda.is_available() else "cpu"
except ImportError:
    device = "cpu"
print(f"Device: {device}")


## Train a Model


In [ ]:
from data.generator import generate_transactions
from models.train import train_pipeline

df = generate_transactions(n_txns=10_000, fraud_rate=0.04, seed=42)
result = train_pipeline(df, model_type="xgb_tabular", split_strategy="temporal", seed=42)
print("Model trained successfully")
for k, v in sorted(result["raw_metrics"].items()):
    print(f"  {k}: {v:.4f}")


## Generate Predictions and Record Decisions


In [ ]:
import json
import numpy as np
from features.engine import build_features, get_feature_matrix
from models.baseline_xgb import TabularXGBoost
from models.calibrate import IsotonicCalibrator
from models.conformal import ConformalPredictor
from replay.recorder import DecisionRecorder
from replay.hashing import decision_hash
import uuid
from pathlib import Path

model = TabularXGBoost.load()
calibrator = IsotonicCalibrator.load()
cp = ConformalPredictor.load()

# Score a few test transactions
test_txns = generate_transactions(n_txns=5, fraud_rate=0.4, seed=99)
test_feat = build_features(test_txns)
X = get_feature_matrix(test_feat).to_numpy().astype(np.float32)

raw_scores = model.predict_proba(X)
calibrated = calibrator.calibrate(raw_scores)
conformal_results = cp.predict(calibrated)

recorder = DecisionRecorder()
decision_ids = []

for i in range(len(test_txns)):
    dec_id = f"DEC_{uuid.uuid4().hex[:16].upper()}"
    prediction = {
        "decision_id": dec_id,
        "tx_id": test_txns["tx_id"][i],
        "raw_score": float(raw_scores[i]),
        "calibrated_score": float(calibrated[i]),
        "confidence_band": conformal_results[i]["confidence_band"],
        "model_type": "xgb_tabular",
    }
    recorder.record_transaction(test_txns["tx_id"][i], test_txns.row(i, named=True))
    recorder.record_features(test_txns["tx_id"][i], X[i].tolist())
    recorder.record_prediction(prediction)
    decision_ids.append(dec_id)
    print(f"  {dec_id}: score={calibrated[i]:.3f} band={conformal_results[i]['confidence_band']}")

print(f"\nRecorded {len(decision_ids)} decisions")


## Generate Audit Reports


In [ ]:
from explain.report import generate_report, report_to_markdown

for dec_id in decision_ids:
    pred = recorder.get_prediction(dec_id)
    report = generate_report(pred)
    md_report = report_to_markdown(report)
    recorder.record_audit_report(dec_id, md_report, report)
    
    print(f"\n{'='*60}")
    print(f"Decision: {dec_id}")
    print(f"{'='*60}")
    print(md_report[:500])
    print("...")


## Deterministic Replay

Replay verifies that a stored decision produces the same hash when re-computed.


In [ ]:
from replay.replayer import ReplayEngine

engine = ReplayEngine(recorder)
for dec_id in decision_ids[:2]:
    result = engine.replay(dec_id)
    match_str = "MATCH" if result["matched"] else "MISMATCH"
    print(f"  Replay {dec_id}: {match_str}")
    print(f"    Stored score: {result['stored_prediction']['calibrated_score']:.4f}")
    print(f"    Band: {result['stored_prediction']['confidence_band']}")


## Decision Lineage


In [ ]:
from replay.lineage import LineageTracker

tracker = LineageTracker(recorder)
for dec_id in decision_ids[:1]:
    lineage = tracker.get_lineage(dec_id)
    print(json.dumps(lineage, indent=2, default=str))


## PII Redaction


In [ ]:
from audit.redact import redact_report, redact_markdown

pred = recorder.get_prediction(decision_ids[0])
report = generate_report(pred)
md_report = report_to_markdown(report)

redacted_md = redact_markdown(md_report)
print("=== Original (excerpt) ===")
print(md_report[:300])
print("\n=== Redacted (excerpt) ===")
print(redacted_md[:300])


## Summary

This notebook demonstrated the full Rift audit lifecycle:
1. Train a model
2. Score transactions and record decisions (SHA-256 hashed)
3. Generate plain-English audit reports with SHAP drivers
4. Replay decisions for deterministic verification
5. Trace full decision lineage
6. Redact PII for external distribution
